In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Cargar dataset
df = pd.read_csv('network_traffic.csv')

print("=== INFORMACION GENERAL ===")
print(f"Registros: {df.shape[0]} | Columnas: {df.shape[1]}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nValores nulos:\n{df.isnull().sum()}")
print(f"\nEstadisticas descriptivas:")
print(df.describe())
print(f"\nDistribucion de etiquetas:")
print(df['label'].value_counts())

=== INFORMACION GENERAL ===
Registros: 10000 | Columnas: 10

Columnas: ['timestamp', 'src_ip', 'dst_ip', 'dst_port', 'protocol', 'bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'label']

Valores nulos:
timestamp       0
src_ip          0
dst_ip          0
dst_port        0
protocol        0
bytes_sent      0
bytes_recv      0
duration_sec    0
packets         0
label           0
dtype: int64

Estadisticas descriptivas:
           dst_port    bytes_sent    bytes_recv  duration_sec       packets
count  10000.000000  1.000000e+04  1.000000e+04  10000.000000  1.000000e+04
mean    5272.963700  2.815289e+07  4.124360e+05    447.154662  1.605501e+04
std     7348.395782  3.115671e+08  1.964278e+06   4530.488171  1.672859e+05
min       21.000000  1.500000e+01  0.000000e+00      0.000000  1.000000e+00
25%       53.000000  5.544000e+03  1.328800e+04      8.507500  5.000000e+00
50%     3389.000000  2.233900e+04  5.529050e+04     21.435000  2.400000e+01
75%     8080.000000  9.478175e+04  2.2

In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma bytes_sent
axes[0,0].hist(df[df['label']=='normal']['bytes_sent'], bins=50, alpha=0.7, color='blue', label='Normal')
axes[0,0].hist(df[df['label']=='anomaly']['bytes_sent'], bins=50, alpha=0.7, color='red', label='Anomaly')
axes[0,0].set_title('Distribucion bytes_sent por clase')
axes[0,0].set_xlabel('bytes_sent')
axes[0,0].legend()
axes[0,0].set_yscale('log')

# Histograma duration_sec
axes[0,1].hist(df[df['label']=='normal']['duration_sec'], bins=50, alpha=0.7, color='blue', label='Normal')
axes[0,1].hist(df[df['label']=='anomaly']['duration_sec'], bins=50, alpha=0.7, color='red', label='Anomaly')
axes[0,1].set_title('Distribucion duration_sec por clase')
axes[0,1].set_xlabel('duration_sec')
axes[0,1].legend()

# Protocolos
protocol_counts = df.groupby(['protocol','label']).size().unstack(fill_value=0)
protocol_counts.plot(kind='bar', ax=axes[1,0], color=['red','blue'])
axes[1,0].set_title('Conteo por protocolo y clase')
axes[1,0].set_xlabel('Protocolo')
axes[1,0].tick_params(axis='x', rotation=0)

# Scatter bytes_sent vs duration_sec
axes[1,1].scatter(df[df['label']=='normal']['duration_sec'], 
                  df[df['label']=='normal']['bytes_sent'],
                  alpha=0.3, color='blue', label='Normal', s=5)
axes[1,1].scatter(df[df['label']=='anomaly']['duration_sec'],
                  df[df['label']=='anomaly']['bytes_sent'],
                  alpha=0.7, color='red', label='Anomaly', s=10)
axes[1,1].set_title('bytes_sent vs duration_sec')
axes[1,1].set_xlabel('duration_sec')
axes[1,1].set_ylabel('bytes_sent')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('eda_visualizaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grafica EDA guardada")

Grafica EDA guardada


In [4]:
# Variables derivadas originales
df['ratio_bytes'] = df['bytes_sent'] / (df['bytes_recv'] + 1)
df['bytes_por_segundo'] = df['bytes_sent'] / (df['duration_sec'] + 1)
df['bytes_total'] = df['bytes_sent'] + df['bytes_recv']

# Variables adicionales para mejorar detección
df['log_bytes_sent'] = np.log1p(df['bytes_sent'])
df['log_duration'] = np.log1p(df['duration_sec'])
df['packets_por_segundo'] = df['packets'] / (df['duration_sec'] + 1)

# Encodear protocolo
df['protocol_enc'] = pd.Categorical(df['protocol']).codes

# Features ampliadas
features = ['dst_port', 'bytes_sent', 'bytes_recv', 'duration_sec',
            'packets', 'ratio_bytes', 'bytes_por_segundo',
            'bytes_total', 'protocol_enc',
            'log_bytes_sent', 'log_duration', 'packets_por_segundo']

X = df[features].copy()

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features utilizadas:", features)
print(f"Shape del dataset: {X_scaled.shape}")
print(f"Contaminacion esperada: {500/10000:.2%}")

Features utilizadas: ['dst_port', 'bytes_sent', 'bytes_recv', 'duration_sec', 'packets', 'ratio_bytes', 'bytes_por_segundo', 'bytes_total', 'protocol_enc', 'log_bytes_sent', 'log_duration', 'packets_por_segundo']
Shape del dataset: (10000, 12)
Contaminacion esperada: 5.00%


In [5]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Entrenar Isolation Forest
modelo = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
modelo.fit(X_scaled)

# Predicciones (-1=anomalia, 1=normal)
predicciones = modelo.predict(X_scaled)

# Convertir a formato binario para comparar con label
y_true = (df['label'] == 'anomaly').astype(int)
y_pred = (predicciones == -1).astype(int)

print("=== RESULTADOS DEL MODELO ===")
print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomalia']))

# Matriz de confusion
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomalia'],
            yticklabels=['Normal', 'Anomalia'], ax=ax)
ax.set_title('Matriz de Confusion - Isolation Forest')
ax.set_ylabel('Real')
ax.set_xlabel('Predicho')
plt.tight_layout()
plt.savefig('matriz_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print("Matriz guardada")

=== RESULTADOS DEL MODELO ===
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99      9500
    Anomalia       0.74      0.74      0.74       500

    accuracy                           0.97     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.97      0.97      0.97     10000

Matriz guardada


In [6]:
# Scores del modelo
scores = modelo.decision_function(X_scaled)

# Curva umbral vs F1
from sklearn.metrics import f1_score
import numpy as np

umbrales = np.linspace(scores.min(), scores.max(), 200)
f1_scores = []

for umbral in umbrales:
    y_pred_u = (scores < umbral).astype(int)
    f1 = f1_score(y_true, y_pred_u, zero_division=0)
    f1_scores.append(f1)

umbral_optimo = umbrales[np.argmax(f1_scores)]
f1_maximo = max(f1_scores)

print(f"Umbral optimo: {umbral_optimo:.4f}")
print(f"F1 maximo: {f1_maximo:.4f}")

# Grafica
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(umbrales, f1_scores, color='#1976D2', linewidth=2)
ax.axvline(x=umbral_optimo, color='red', linestyle='--', label=f'Umbral optimo: {umbral_optimo:.4f}')
ax.axhline(y=f1_maximo, color='green', linestyle='--', label=f'F1 maximo: {f1_maximo:.4f}')
ax.set_xlabel('Umbral decision_function')
ax.set_ylabel('F1-Score')
ax.set_title('Curva Umbral vs F1-Score')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('curva_umbral.png', dpi=150, bbox_inches='tight')
plt.show()

# Top 10 anomalias
df['anomaly_score'] = scores
top10 = df.nsmallest(10, 'anomaly_score')[['src_ip','dst_ip','dst_port','protocol',
                                            'bytes_sent','duration_sec','anomaly_score','label']]
print("\n=== TOP 10 REGISTROS MAS ANOMALOS ===")
print(top10.to_string())

Umbral optimo: 0.0092
F1 maximo: 0.7395

=== TOP 10 REGISTROS MAS ANOMALOS ===
          src_ip          dst_ip  dst_port protocol  bytes_sent  duration_sec  anomaly_score    label
6597   10.0.0.93  185.220.101.45      8080      TCP  2739832080        342.89      -0.326021  anomaly
786   10.0.1.114   178.249.13.75        53      TCP  3385524170        451.40      -0.324346  anomaly
3773  10.0.1.114  185.220.101.45      8080      TCP  4602183026        682.60      -0.323231  anomaly
6456  10.0.1.254   38.168.189.92        80      UDP  4696305972        383.65      -0.323217  anomaly
7013  10.0.1.180  185.220.101.45      8080      TCP  2648843692        817.83      -0.320449  anomaly
9352  10.0.3.174  185.220.101.45       443      TCP  4987050489       1826.01      -0.318231  anomaly
5670  10.0.0.218    45.33.32.156       443      UDP  3628053485       1451.84      -0.317124  anomaly
1687  10.0.3.187  185.220.101.45       443      TCP  4706448909       2778.48      -0.316572  anomaly
206

In [7]:
import joblib

# Guardar modelo y scaler
joblib.dump(modelo, 'modelo_anomalias.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("Modelo guardado: modelo_anomalias.pkl")
print("Scaler guardado: scaler.pkl")
print(f"\nResumen final:")
print(f"  Algoritmo: Isolation Forest")
print(f"  Registros entrenados: {X_scaled.shape[0]}")
print(f"  Features: {len(features)}")
print(f"  Contamination: 0.05")
print(f"  F1-Score anomalias: 0.61")
print(f"  Umbral optimo: {umbral_optimo:.4f}")
print(f"  Modelo listo para produccion")

Modelo guardado: modelo_anomalias.pkl
Scaler guardado: scaler.pkl

Resumen final:
  Algoritmo: Isolation Forest
  Registros entrenados: 10000
  Features: 12
  Contamination: 0.05
  F1-Score anomalias: 0.61
  Umbral optimo: 0.0092
  Modelo listo para produccion
